# Introduction to Natural Language Processing

Human language is our primary tool for communication — rich, flexible, and deeply contextual. **Natural Language Processing (NLP)** is the field of artificial intelligence concerned with enabling computers to understand, interpret, and generate human language. Every time you ask a voice assistant a question, use autocomplete on your phone, translate a webpage, or chat with a language model, NLP is at work.

Language is deceptively difficult for machines. The same word can mean different things in different contexts. Sarcasm inverts literal meaning. A single sentence can have multiple valid interpretations. Languages differ in grammar, script, and idiom. NLP must bridge the gap between messy, ambiguous human expression and the structured computations that machines perform.

This notebook introduces what NLP is, why language is hard, the major tasks and challenges in the field, how NLP systems have evolved over decades, and the building blocks of a modern language processing pipeline.

In [ ]:
import re
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter

np.set_printoptions(precision=4, suppress=True)
plt.style.use("seaborn-v0_8-whitegrid")

---

## 1. What Is Natural Language Processing?

**Natural language** is any language that evolved organically among humans — English, Hindi, Mandarin, Arabic, and thousands of others. This contrasts with **formal languages** like programming languages, which have precise, unambiguous grammars designed by specification.

**Natural Language Processing** combines linguistics, computer science, and machine learning to build systems that work with human language. The scope is broad:

- **Understanding** — extract meaning from text or speech (What is the sentiment? Who did what to whom?).
- **Generation** — produce fluent, coherent text (summaries, translations, dialogue).
- **Transformation** — convert language from one form to another (translation, paraphrasing, speech-to-text).

NLP sits at the intersection of AI and human communication. Its applications span search engines, customer support chatbots, medical record analysis, legal document review, accessibility tools, and the large language models that have transformed how people interact with software.

---

## 2. NLP, NLU, and NLG

Three related terms appear frequently:

| Term | Focus | Example |
|------|-------|--------|
| **NLP** | All processing of human language | Umbrella term for the entire field |
| **NLU** (Natural Language Understanding) | Comprehending meaning from input | Sentiment analysis, intent detection |
| **NLG** (Natural Language Generation) | Producing human-like text | Summarization, chatbot responses |

Most real systems combine both. A virtual assistant must **understand** your question (NLU) and **generate** a helpful answer (NLG). Modern large language models perform both in a single architecture — they read context and produce continuations that demonstrate understanding and generation simultaneously.

---

## 3. Why Language Is Hard for Computers

Unlike images (pixels arranged in space) or tabular data (fixed columns of numbers), language presents unique challenges:

**Ambiguity at every level.** The word *"bank"* can mean a financial institution or a river edge. The sentence *"I saw the man with the telescope"* has at least two structural interpretations. Machines must resolve ambiguity using context.

**Context dependence.** *"It's cold in here"* might be a statement of fact or an implicit request to close a window. Meaning depends on who is speaking, the situation, and prior conversation.

**Variability.** People express the same idea in countless ways. *"What's the weather?"*, *"Will it rain today?"*, and *"Should I bring an umbrella?"* can all seek similar information.

**Non-standard language.** Social media introduces abbreviations (*"u"*, *"tbh"*), misspellings, emojis, and code-switching between languages in a single sentence.

**World knowledge.** Understanding *"The trophy didn't fit in the suitcase because it was too big"* requires knowing that trophies are physical objects and that *"it"* likely refers to the trophy, not the suitcase.

**Scale.** English alone has hundreds of thousands of words. Multilingual systems must handle dozens of scripts, grammars, and cultural conventions.

In [ ]:
# Lexical ambiguity: same word, different meanings
ambiguous_words = {
    "bank": [
        "She deposited money at the bank.",
        "We sat on the river bank.",
    ],
    "bat": [
        "He swung the bat and hit a home run.",
        "A bat flew out of the cave at dusk.",
    ],
    "light": [
        "Turn on the light, it's dark in here.",
        "This suitcase is surprisingly light.",
    ],
}

for word, sentences in ambiguous_words.items():
    print(f"\n'{word}':")
    for s in sentences:
        print(f"  • {s}")

---

## 4. Levels of Linguistic Analysis

Linguists and NLP systems analyze language at multiple levels, from surface form to deep meaning:

| Level | Question Answered | Example |
|-------|-------------------|--------|
| **Phonetics/Phonology** | What sounds are produced? | /kæt/ for "cat" |
| **Morphology** | How are words built? | "unhappiness" = un- + happy + -ness |
| **Syntax** | How are words structured into sentences? | Subject-verb-object order |
| **Semantics** | What do words and sentences mean? | "Dog bites man" vs "Man bites dog" |
| **Pragmatics** | What is meant in context? | Sarcasm, implicature, politeness |
| **Discourse** | How do sentences connect across a text? | Coreference: "John arrived. He was late." |

Different NLP tasks operate at different levels. Spell checking is morphological. Part-of-speech tagging is syntactic. Sentiment analysis is semantic-pragmatic. Dialogue systems must handle discourse-level coherence across many turns.

---

## 5. Core NLP Tasks

The field encompasses dozens of tasks. Here are the most important:

**Text classification** — Assign a label to a document. Spam detection, sentiment analysis, topic categorization.

**Named Entity Recognition (NER)** — Find and classify entities: people, organizations, locations, dates. *"Apple announced a new iPhone in Cupertino on Tuesday."*

**Part-of-Speech (POS) tagging** — Label each word with its grammatical role: noun, verb, adjective, etc.

**Parsing** — Determine the syntactic tree structure of a sentence.

**Machine translation** — Convert text from one language to another.

**Question answering** — Given a question and a context, extract or generate the answer.

**Summarization** — Condense a long document into a shorter version preserving key information.

**Information extraction** — Pull structured data from unstructured text (events, relations, facts).

**Speech recognition** — Convert spoken audio to text.

**Text generation** — Produce coherent text: stories, code, dialogue, completions.

**Coreference resolution** — Determine which expressions refer to the same entity across a document.

In [ ]:
# Sample document with multiple NLP tasks illustrated
document = (
    "On March 15, 2024, Tesla CEO Elon Musk announced that the company "
    "would open a new factory in Berlin. Investors reacted positively, "
    "and the stock rose 5%. Analysts at Goldman Sachs praised the move."
)

tasks = {
    "NER": ["Tesla", "Elon Musk", "Berlin", "Goldman Sachs", "March 15, 2024"],
    "Sentiment": "Positive (investors reacted positively, stock rose)",
    "Relation": "Elon Musk → CEO → Tesla; Tesla → opens → factory → Berlin",
    "Summary": "Tesla to open Berlin factory; stock up 5%.",
}

print("Document:")
print(f'  "{document}"\n')
for task, result in tasks.items():
    print(f"{task:12s}: {result}")

---

## 6. A Brief History of NLP

NLP has evolved through distinct eras, each driven by new ideas and available data:

**1950s–1980s: Rule-based systems.** Researchers wrote hand-crafted grammars and dictionaries. ELIZA (1966) mimicked a therapist using pattern-matching rules. Systems were interpretable but brittle — they failed on language outside their rules.

**1990s–2000s: Statistical NLP.** With more digital text available, researchers shifted to probabilistic models. N-gram language models estimated word probabilities. Hidden Markov Models powered speech recognition and POS tagging. Machine learning classifiers used hand-engineered features.

**2010s: Neural NLP.** Word embeddings (Word2Vec, GloVe) represented words as dense vectors capturing semantic similarity. Recurrent networks and later Transformers learned representations directly from data. Sequence-to-sequence models revolutionized translation.

**2018–present: Pre-trained language models.** BERT, GPT, and their successors are trained on massive text corpora and fine-tuned for specific tasks. Large language models with billions of parameters demonstrate remarkable generalization — often performing new tasks from instructions alone, without task-specific training.

Each era built on the last. Modern systems still use tokenization, parsing concepts, and evaluation metrics developed decades ago — but the representations and models have transformed completely.

---

## 7. Rule-Based vs Machine Learning vs Deep Learning

| Approach | How It Works | Strengths | Weaknesses |
|----------|-------------|-----------|------------|
| **Rule-based** | Human-written patterns and grammars | Interpretable, no training data needed | Brittle, doesn't scale |
| **Classical ML** | Features + classifier (SVM, logistic regression) | Works with small data, fast | Requires feature engineering |
| **Deep learning** | Neural networks learn features from raw text | State-of-the-art accuracy, end-to-end | Needs large data and compute |
| **Pre-trained LLMs** | Fine-tune or prompt large models | Generalizes across tasks | Expensive, harder to control |

Production systems often combine approaches: a rule-based filter catches obvious cases, a neural model handles the rest, and business logic enforces constraints the model cannot guarantee.

In [ ]:
# Rule-based sentiment: simple keyword matching
POSITIVE_WORDS = {"good", "great", "excellent", "love", "amazing", "happy", "wonderful", "fantastic"}
NEGATIVE_WORDS = {"bad", "terrible", "awful", "hate", "horrible", "sad", "poor", "worst"}

def rule_based_sentiment(text):
    words = set(re.findall(r"\b\w+\b", text.lower()))
    pos = len(words & POSITIVE_WORDS)
    neg = len(words & NEGATIVE_WORDS)
    if pos > neg:
        return "positive"
    elif neg > pos:
        return "negative"
    return "neutral"

test_reviews = [
    "This product is absolutely amazing, I love it!",
    "Terrible quality, worst purchase ever.",
    "The movie was good but the ending was bad.",
    "Great, another meeting that could have been an email.",  # sarcasm
    "Not bad, not great, just okay.",
]

print(f"{'Review':<55s} {'Prediction':>10s}")
print("-" * 67)
for review in test_reviews:
    pred = rule_based_sentiment(review)
    print(f"{review[:53]:<55s} {pred:>10s}")

The rule-based classifier works on straightforward cases but fails on mixed sentiment, negation (*"not bad"*), and sarcasm. This motivates machine learning approaches that learn patterns from labeled examples rather than relying on fixed word lists.

---

## 8. The NLP Pipeline

Most NLP systems follow a pipeline — a sequence of processing steps that transform raw text into useful output:

```
Raw Text
    ↓
Text Cleaning (remove HTML, normalize whitespace)
    ↓
Tokenization (split into words/subwords)
    ↓
Normalization (lowercasing, stemming, lemmatization)
    ↓
Feature Extraction (BoW, TF-IDF, embeddings)
    ↓
Model (classifier, tagger, translator, generator)
    ↓
Post-processing (format output, apply business rules)
    ↓
Result
```

End-to-end neural models blur these steps. A Transformer receives raw tokens (after tokenization) and produces task output directly. But the conceptual pipeline remains useful — preprocessing quality still affects downstream performance, and understanding each stage helps debug failures.

---

## 9. Representing Text for Machines

Computers process numbers, not words. Converting text to numerical form is a foundational step:

**Discrete representations:**
- **One-hot encoding** — each word is a vector with a single 1 and all other positions 0. Vocabulary-sized, sparse, no notion of similarity.
- **Bag of Words (BoW)** — count word occurrences in a document. Loses word order.
- **TF-IDF** — weight words by frequency in a document relative to the corpus. Downweights common words like *"the"*.

**Dense representations (embeddings):**
- Each word is a low-dimensional vector (e.g., 300 dimensions) where similar words have similar vectors. *"king"* − *"man"* + *"woman"* ≈ *"queen"*.

**Contextual representations:**
- Modern models produce different vectors for the same word in different contexts. *"bank"* in a financial sentence gets a different representation than *"bank"* in a river sentence.

The progression from sparse one-hot vectors to dense contextual embeddings mirrors the field's evolution from counting words to understanding meaning.

In [ ]:
# Basic text statistics: the first step in understanding any corpus
corpus = [
    "Natural language processing enables computers to understand human language.",
    "Machine learning models learn patterns from text data.",
    "Deep learning has transformed natural language processing.",
    "Transformers are the foundation of modern language models.",
    "Text preprocessing is the first step in any NLP pipeline.",
]

all_words = []
for doc in corpus:
    words = re.findall(r"\b[a-z]+\b", doc.lower())
    all_words.extend(words)

word_counts = Counter(all_words)
vocab_size = len(word_counts)
total_tokens = len(all_words)
avg_doc_len = total_tokens / len(corpus)

print(f"Documents:        {len(corpus)}")
print(f"Total tokens:     {total_tokens}")
print(f"Vocabulary size:  {vocab_size}")
print(f"Avg doc length:   {avg_doc_len:.1f} words")
print(f"\nTop 10 words:")
for word, count in word_counts.most_common(10):
    print(f"  {word:20s} {count}")

In [ ]:
# Word frequency distribution (Zipf's law preview)
freqs = sorted(word_counts.values(), reverse=True)
ranks = np.arange(1, len(freqs) + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

words, counts = zip(*word_counts.most_common(10))
axes[0].barh(words[::-1], counts[::-1], color="steelblue")
axes[0].set_xlabel("Count")
axes[0].set_title("Top 10 word frequencies")

axes[1].loglog(ranks, freqs, "o", markersize=4, color="coral")
axes[1].set_xlabel("Rank")
axes[1].set_ylabel("Frequency")
axes[1].set_title("Zipf's law: frequency ∝ 1/rank")

plt.tight_layout()
plt.show()

**Zipf's law** observes that in natural language, the frequency of a word is inversely proportional to its rank. A few words (*"the"*, *"of"*, *"and"*) appear very often; most words appear rarely. This skewed distribution affects how we represent text — common words carry little discriminative information, while rare words may appear only once.

---

## 10. Evaluating NLP Systems

Evaluation is task-specific. There is no single metric for all of NLP.

| Task | Common Metrics |
|------|---------------|
| Classification | Accuracy, precision, recall, F1-score |
| NER / POS tagging | Token-level accuracy, F1 per entity type |
| Machine translation | BLEU, chrF, human evaluation |
| Summarization | ROUGE (overlap with reference), human evaluation |
| Generation | Perplexity, human preference, task-specific rubrics |
| Question answering | Exact match, F1 on answer span |

**Automatic metrics** are fast and reproducible but imperfect. BLEU measures n-gram overlap with reference translations — a fluent but different translation may score poorly. **Human evaluation** remains the gold standard for generation quality but is expensive and subjective.

For classification tasks, **F1-score** balances precision (of predicted positives, how many are correct?) and recall (of actual positives, how many did we find?). It is preferred over accuracy when classes are imbalanced — a spam filter that labels everything as "not spam" has high accuracy but zero recall on spam.

In [ ]:
# Precision, recall, F1 for a binary classification example
def prf1(tp, fp, fn):
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
    return precision, recall, f1

# Spam detector: 100 emails, 5 spam. Model catches 4 spam but has 6 false alarms.
tp, fp, fn = 4, 6, 1
tn = 95 - fp  # not-spam correctly ignored
precision, recall, f1 = prf1(tp, fp, fn)
accuracy = (tp + tn) / 100

print(f"Accuracy:  {accuracy:.2f}  (misleading — 95% of emails aren't spam)")
print(f"Precision: {precision:.2f}  ({tp}/{tp+fp} flagged emails are actually spam)")
print(f"Recall:    {recall:.2f}  ({tp}/{tp+fn} spam emails caught)")
print(f"F1:        {f1:.2f}  (balanced measure)")

---

## 11. Real-World Applications

NLP powers systems people interact with daily:

**Search** — Google, Bing, and enterprise search engines parse queries, match documents, and rank results using NLP at every stage.

**Customer support** — Chatbots and ticket routing systems classify intent, extract entities, and generate responses.

**Healthcare** — Clinical note analysis, drug interaction detection, and medical coding from free-text records.

**Finance** — Sentiment analysis on news and social media for trading signals; fraud detection from transaction descriptions.

**Legal** — Contract analysis, case law search, and due diligence document review.

**Accessibility** — Speech-to-text, text-to-speech, and real-time captioning.

**Content moderation** — Detecting hate speech, misinformation, and policy violations at scale.

**Education** — Automated essay scoring, language learning tools, and tutoring systems.

---

## 12. Large Language Models and Modern NLP

The current era of NLP is defined by **large language models (LLMs)** — neural networks with billions of parameters trained on vast text corpora. Models like GPT, Claude, Gemini, and Llama can:

- Answer questions across domains without task-specific training.
- Summarize, translate, and rewrite text from natural language instructions.
- Write code, analyze data, and engage in multi-turn dialogue.
- Perform many NLP tasks through **prompting** rather than fine-tuning.

This shift changes the practitioner's workflow. Instead of building a custom sentiment classifier from scratch, you may prompt an LLM with examples (*few-shot learning*) or fine-tune a pre-trained model on a small labeled dataset. The fundamentals — tokenization, embeddings, evaluation — remain essential for understanding, debugging, and building reliable systems.

LLMs also introduce new challenges: hallucination (generating plausible but false information), bias inherited from training data, high computational cost, and difficulty guaranteeing factual accuracy in high-stakes domains.

---

## 13. The Python NLP Ecosystem

Python dominates NLP research and production. Key libraries:

| Library | Purpose |
|---------|--------|
| **NLTK** | Teaching, linguistic resources, basic NLP tools |
| **spaCy** | Production-grade NLP pipeline (tokenization, NER, parsing) |
| **Hugging Face Transformers** | Pre-trained models (BERT, GPT, T5) and tokenizers |
| **scikit-learn** | Classical ML on text features (BoW, TF-IDF + classifiers) |
| **Gensim** | Topic modeling, Word2Vec, document similarity |
| **regex / re** | Pattern matching for text cleaning and extraction |

For learning fundamentals, starting with Python's built-in `re` module and scikit-learn builds intuition before moving to neural libraries. For production, spaCy and Hugging Face provide battle-tested pipelines.

---

## 14. Ethical Considerations

NLP systems interact with human language — and human language carries bias, harm, and privacy concerns.

**Bias.** Models trained on web text inherit societal biases. A resume screening model may disadvantage certain names. Translation systems may default to masculine pronouns. Evaluation must include fairness across demographic groups.

**Privacy.** NLP systems process personal communications, medical records, and financial documents. Data handling, anonymization, and compliance with regulations (GDPR, HIPAA) are mandatory.

**Misinformation.** Text generation models can produce convincing false content at scale. Detection and watermarking are active research areas.

**Environmental cost.** Training large language models consumes significant energy. Efficient architectures and smaller task-specific models are important for sustainability.

Building responsible NLP systems requires considering these factors from the design stage, not as an afterthought.

---

## 15. Summary

**Natural Language Processing** enables computers to work with human language — understanding, generating, and transforming text and speech. Language is inherently ambiguous and context-dependent, making NLP one of the most challenging areas of AI.

The field spans tasks from classification and entity recognition to translation, summarization, and open-ended generation. Systems have evolved from hand-written rules through statistical models to neural networks and large pre-trained language models.

Every NLP system converts text to numerical representations, applies a model, and evaluates output with task-appropriate metrics. The pipeline — cleaning, tokenization, feature extraction, modeling — provides the conceptual framework even as end-to-end neural models simplify implementation.

Understanding these foundations prepares you to preprocess text, build representations, train classifiers, and work with the language models that define modern NLP.